In [5]:
import numpy as np
import random

class SimpleChipEnv:
    def __init__(self, grid_size=6, num_macros=4):
        self.grid_size = grid_size
        self.num_macros = num_macros
        self.reset()

    def calculate_wirelength(self, positions):
        if len(positions) < 2:
            return 0
        total = 0
        for i in range(len(positions) - 1):
            x1, y1 = positions[i]
            x2, y2 = positions[i + 1]
            total += abs(x1 - x2) + abs(y1 - y2)
        return total

    def reset(self):
        self.grid = np.zeros((self.grid_size, self.grid_size))
        self.macros_placed = []
        self.step_count = 0
        return self.grid

    def step(self, action):
        x, y = action
        if 0 <= x < self.grid_size and 0 <= y < self.grid_size and self.grid[x, y] == 0:
            self.grid[x, y] = 1
            self.macros_placed.append((x, y))
            self.step_count += 1
            wirelength = self.calculate_wirelength(self.macros_placed)
            reward = -wirelength
            done = len(self.macros_placed) >= self.num_macros
            return self.grid, reward, done
        else:
            return self.grid, -10, False

class QLearningAgent:
    def __init__(self, env, learning_rate=0.1, discount_factor=0.9, exploration_rate=0.3):
        self.env = env
        self.lr = learning_rate
        self.gamma = discount_factor
        self.epsilon = exploration_rate
        self.q_table = {}

    def get_state_key(self):
        return tuple(sorted(self.env.macros_placed))

    def get_action(self):
        state_key = self.get_state_key()

        if random.random() < self.epsilon:
            x = random.randint(0, self.env.grid_size - 1)
            y = random.randint(0, self.env.grid_size - 1)
            return (x, y)

        if state_key not in self.q_table:
            self.q_table[state_key] = {}

        if len(self.q_table[state_key]) == 0:
            x = random.randint(0, self.env.grid_size - 1)
            y = random.randint(0, self.env.grid_size - 1)
            return (x, y)

        best_action = max(self.q_table[state_key], key=self.q_table[state_key].get)
        return best_action

    def update(self, state, action, reward, next_state, done):
        state_key = tuple(sorted(state))
        next_state_key = tuple(sorted(next_state))

        if state_key not in self.q_table:
            self.q_table[state_key] = {}
        if action not in self.q_table[state_key]:
            self.q_table[state_key][action] = 0

        max_future_q = 0
        if next_state_key in self.q_table and len(self.q_table[next_state_key]) > 0:
            max_future_q = max(self.q_table[next_state_key].values())

        old_q = self.q_table[state_key][action]
        new_q = old_q + self.lr * (reward + self.gamma * max_future_q - old_q)
        self.q_table[state_key][action] = new_q

print("=" * 50)
print("Training Q-Learning Agent - 500 Episodes")
print("=" * 50)

env = SimpleChipEnv(grid_size=6, num_macros=4)
agent = QLearningAgent(env)

num_episodes = 500
episode_rewards = []

for episode in range(num_episodes):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.get_action()
        next_state, reward, done = env.step(action)
        agent.update(env.macros_placed[:-1], action, reward, env.macros_placed, done)
        total_reward += reward

    episode_rewards.append(total_reward)

    if (episode + 1) % 50 == 0:
        avg_reward = np.mean(episode_rewards[-50:])
        print(f"Episode {episode+1}: Average Reward = {avg_reward:.2f}")

print("\nTraining Complete!")
print(f"Best reward achieved: {max(episode_rewards)}")
print(f"Average reward over last 50 episodes: {np.mean(episode_rewards[-50:]):.2f}")

Training Q-Learning Agent - 500 Episodes
Episode 50: Average Reward = -21.82
Episode 100: Average Reward = -22.36
Episode 150: Average Reward = -24.48
Episode 200: Average Reward = -24.34
Episode 250: Average Reward = -21.98
Episode 300: Average Reward = -22.40
Episode 350: Average Reward = -23.24
Episode 400: Average Reward = -24.06
Episode 450: Average Reward = -21.36
Episode 500: Average Reward = -21.34

Training Complete!
Best reward achieved: -9
Average reward over last 50 episodes: -21.34


In [4]:
!pip install torch gymnasium numpy matplotlib

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

# ============================================
# DQN Neural Network
# ============================================
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.network(x)

# ============================================
# DQN Agent (Fixed)
# ============================================
class DQNAgent:
    def __init__(self, env, learning_rate=0.001, discount_factor=0.9,
                 exploration_rate=1.0, exploration_decay=0.995,
                 exploration_min=0.01, memory_size=2000, batch_size=32):
        self.env = env
        self.lr = learning_rate
        self.gamma = discount_factor
        self.epsilon = exploration_rate
        self.epsilon_decay = exploration_decay
        self.epsilon_min = exploration_min
        self.batch_size = batch_size
        self.memory = deque(maxlen=memory_size)

        self.input_dim = env.grid_size * env.grid_size
        self.output_dim = env.grid_size * env.grid_size

        self.policy_net = DQN(self.input_dim, self.output_dim)
        self.target_net = DQN(self.input_dim, self.output_dim)
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=self.lr)

        self.target_net.load_state_dict(self.policy_net.state_dict())

    def get_state_tensor(self, grid):
        """Convert grid to tensor"""
        return torch.FloatTensor(grid.flatten()).unsqueeze(0)

    def get_action(self, grid):
        """Choose action using epsilon-greedy"""
        if random.random() < self.epsilon:
            return random.randint(0, self.env.grid_size * self.env.grid_size - 1)

        state_tensor = self.get_state_tensor(grid)
        with torch.no_grad():
            q_values = self.policy_net(state_tensor)
        return torch.argmax(q_values).item()

    def remember(self, state, action, reward, next_state, done):
        """Store experience in memory"""
        self.memory.append((state, action, reward, next_state, done))

    def replay(self):
        """Train on random batch from memory"""
        if len(self.memory) < self.batch_size:
            return

        batch = random.sample(self.memory, self.batch_size)

        for state, action, reward, next_state, done in batch:
            state_tensor = self.get_state_tensor(state)
            next_state_tensor = self.get_state_tensor(next_state)

            # Current Q-value
            current_q = self.policy_net(state_tensor)[0][action]

            # Target Q-value
            if done:
                target_q = torch.tensor(reward, dtype=torch.float32)
            else:
                with torch.no_grad():
                    next_q = self.target_net(next_state_tensor).max()
                target_q = torch.tensor(reward, dtype=torch.float32) + self.gamma * next_q

            # Loss (fixed: convert target_q to same dtype as current_q)
            loss = nn.MSELoss()(current_q, target_q)

            # Backpropagation
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

        # Decay exploration rate
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def update_target_network(self):
        """Copy policy network weights to target network"""
        self.target_net.load_state_dict(self.policy_net.state_dict())

# ============================================
# Simple Environment (compatible with DQN)
# ============================================
class SimpleChipEnvDQN:
    def __init__(self, grid_size=6, num_macros=4):
        self.grid_size = grid_size
        self.num_macros = num_macros
        self.reset()

    def calculate_wirelength(self, positions):
        if len(positions) < 2:
            return 0
        total = 0
        for i in range(len(positions) - 1):
            x1, y1 = positions[i]
            x2, y2 = positions[i + 1]
            total += abs(x1 - x2) + abs(y1 - y2)
        return total

    def reset(self):
        self.grid = np.zeros((self.grid_size, self.grid_size))
        self.macros_placed = []
        self.step_count = 0
        return self.grid.copy()

    def step(self, action_index):
        x = action_index // self.grid_size
        y = action_index % self.grid_size

        if self.grid[x, y] == 0:
            self.grid[x, y] = 1
            self.macros_placed.append((x, y))
            self.step_count += 1

            wirelength = self.calculate_wirelength(self.macros_placed)
            reward = -wirelength
            done = len(self.macros_placed) >= self.num_macros

            return self.grid.copy(), reward, done
        else:
            return self.grid.copy(), -10, False

# ============================================
# Train DQN Agent
# ============================================
print("=" * 50)
print("Training DQN Agent - 200 Episodes")
print("=" * 50)

env = SimpleChipEnvDQN(grid_size=6, num_macros=4)
agent = DQNAgent(env)

num_episodes = 200
episode_rewards = []

for episode in range(num_episodes):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.get_action(state)
        next_state, reward, done = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        agent.replay()
        state = next_state
        total_reward += reward

    episode_rewards.append(total_reward)

    if (episode + 1) % 10 == 0:
        agent.update_target_network()

    if (episode + 1) % 50 == 0:
        avg_reward = np.mean(episode_rewards[-50:])
        print(f"Episode {episode+1}: Average Reward = {avg_reward:.2f}, Epsilon = {agent.epsilon:.3f}")

print("\n" + "=" * 50)
print("Training Complete!")
print(f"Best reward achieved: {max(episode_rewards)}")
print(f"Average reward over last 50 episodes: {np.mean(episode_rewards[-50:]):.2f}")
print("=" * 50)

Training DQN Agent - 200 Episodes
Episode 50: Average Reward = -28.22, Epsilon = 0.384
Episode 100: Average Reward = -21.28, Epsilon = 0.126
Episode 150: Average Reward = -16.50, Epsilon = 0.041
Episode 200: Average Reward = -7.74, Epsilon = 0.015

Training Complete!
Best reward achieved: -6
Average reward over last 50 episodes: -7.74


In [ ]:
print("=" * 50)
print("COMPARISON: Random vs Q-Learning vs DQN")
print("=" * 50)

# Test Random Agent
env = SimpleChipEnvDQN(grid_size=6, num_macros=4)
random_rewards = []
for _ in range(50):
    state = env.reset()
    total = 0
    done = False
    while not done:
        action = random.randint(0, 35)
        next_state, reward, done = env.step(action)
        total += reward
    random_rewards.append(total)
random_avg = np.mean(random_rewards)

# Test Q-Learning (from earlier - use your trained agent or run 100 episodes)
# For comparison, we'll use the average from Q-learning training

# Test DQN (from training above)
dqn_avg = np.mean(episode_rewards[-50:]) if 'episode_rewards' in dir() else -25

print(f"Random Agent Average Reward: {random_avg:.2f}")
print(f"Q-Learning Average Reward (last 50): -20.00 (approx)")
print(f"DQN Average Reward (last 50): {dqn_avg:.2f}")

print("\nImprovement:")
print(f"Q-Learning vs Random: {(random_avg - (-20)) / abs(random_avg) * 100:.1f}% better")
print(f"DQN vs Random: {(random_avg - dqn_avg) / abs(random_avg) * 100:.1f}% better")